# Pipeline KDD — CAR/SICAR

Sistematização de Ciência de Dados II. Ver `plano_trabalho.md` e `RELATORIO.md` para contexto.

## Etapa 1 — Seleção do Dataset

- **Fonte:** Portal oficial do SICAR (consultapublica.car.gov.br/publico/estados/downloads),
  camada "Área do Imóvel", estado da **Bahia (BA)** — baixado manualmente (o download exige
  captcha, não dá pra automatizar) e convertido de `.dbf` para `.csv` com `src/leitor_dbf.py`
  (leitor próprio do formato .dbf, sem geopandas/fiona).
- **Volume:** 1.313.653 linhas (bem acima do mínimo de 100 mil exigido) × 12 colunas — CSV de
  ~184 MB.
- **Colunas:** `cod_tema`, `nom_tema` (constantes — sempre "AREA_IMOVEL"/"Area do Imovel" nesta
  camada), `cod_imovel` (identificador único do imóvel no SICAR), `mod_fiscal` (tamanho em
  módulos fiscais), `num_area` (área total declarada), `ind_status` (situação do cadastro —
  ex. "AT" ativo — **alvo da Etapa 4**), `ind_tipo` (tipo de imóvel, ex. "IRU" rural),
  `des_condic` (descrição textual da condição, ex. "Aguardando analise"), `municipio`,
  `cod_estado`, `dat_criaca`/`dat_atuali` (datas de criação/atualização do registro, formato
  `DD/MM/AAAA` — vêm como string, precisam de `to_date(..., 'dd/MM/yyyy')` na Etapa 2).
- **Justificativa:** CAR/SICAR é o cadastro nacional de imóveis rurais, público e tabular por
  natureza; a Bahia sozinha já ultrapassa em mais de 13x o volume mínimo exigido pelo trabalho,
  e o tema é relevante para o país e a dificuldade de manter seu cadastro rural atualizado devido a sua dimensão territorial.
  `cod_tema`/`nom_tema` não agregam informação (valor único) e serão descartadas na Etapa 2.

In [4]:
# No Windows, o Spark às vezes escolhe o Python errado pra abrir o processo
# "worker" (viramos vítimas disso no diagnóstico do ambiente: dava
# "java.net.SocketException: Connection reset" porque ele tentava usar um
# outro Python instalado na máquina, não este). Forçar PYSPARK_PYTHON pro
# mesmo interpretador que está rodando o notebook resolve.
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("car-sicar-kdd")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark

In [3]:
df = spark.read.csv("../data/raw/AREA_IMOVEL_1.csv", header=True, inferSchema=True)
df.printSchema()
df.count()

root
 |-- cod_tema: string (nullable = true)
 |-- nom_tema: string (nullable = true)
 |-- cod_imovel: string (nullable = true)
 |-- mod_fiscal: double (nullable = true)
 |-- num_area: double (nullable = true)
 |-- ind_status: string (nullable = true)
 |-- ind_tipo: string (nullable = true)
 |-- des_condic: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- cod_estado: string (nullable = true)
 |-- dat_criaca: string (nullable = true)
 |-- dat_atuali: string (nullable = true)



1313653

## Etapa 2 — Ingestão e Pré-processamento com Spark

Tratamento de nulos, tipos, duplicatas, outliers e engenharia de atributos.

**Achados principais:**

- `cod_tema` e `nom_tema` são constantes em todo o dataset (só existe "AREA_IMOVEL" /
  "Area do Imovel") — não carregam informação nenhuma, foram descartadas.
- 33 grupos de `cod_imovel` duplicado (66 linhas no total). Investigando um exemplo,
  o padrão ficou claro pela coluna `des_condic`: cada duplicidade é uma linha com
  `ind_status = "AT"` (ativa) mais uma ou mais linhas com status de cancelamento e
  `des_condic` explicando o motivo — no caso mais comum, "Cancelado por duplicidade".
  Verificado que o padrão se repete em todos os 33 grupos, não só no exemplo. Resolvido
  mantendo a linha `"AT"` de cada grupo (ou a mais recente por `dat_atuali`, quando não
  há nenhuma ativa) — de 1.313.653 linhas para 1.313.620, batendo exatamente com o
  número de `cod_imovel` distintos.
- 8 nulos em `dat_atuali` (nenhum nulo nas demais colunas), preenchidos com o valor de
  `dat_criaca` (assume-se que um imóvel nunca atualizado manteve a data de criação como
  última atualização).
- `dat_criaca` e `dat_atuali` vieram como texto no formato `DD/MM/AAAA` — convertidas
  para o tipo `date` do Spark.
- Não foram encontrados outliers óbvios de área (`num_area`) ou módulo fiscal
  (`mod_fiscal`) negativos ou zerados.
- `ind_status` é fortemente desbalanceado (~99,4% `AT`) — relevante para a Etapa 4
  (modelagem preditiva), vai precisar de tratamento de desbalanceamento (ex. pesos por
  classe, undersampling/oversampling, ou métricas que não sejam só acurácia).


In [5]:
from pyspark.sql import functions as F

# 1) cod_tema/nom_tema são mesmo constantes (sem informação nenhuma)?
df.select("cod_tema", "nom_tema").distinct().show()

# 2) quantos nulos tem em cada coluna?
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 3) cod_imovel é único, ou tem imóvel repetido?
total = df.count()
distintos = df.select("cod_imovel").distinct().count()
print(f"total de linhas: {total} | cod_imovel distintos: {distintos}")

# 4) quais valores existem em ind_status e ind_tipo, e quão comum é cada um?
df.groupBy("ind_status").count().orderBy(F.desc("count")).show()
df.groupBy("ind_tipo").count().orderBy(F.desc("count")).show()

+-----------+--------------+
|   cod_tema|      nom_tema|
+-----------+--------------+
|AREA_IMOVEL|Area do Imovel|
+-----------+--------------+

+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|cod_tema|nom_tema|cod_imovel|mod_fiscal|num_area|ind_status|ind_tipo|des_condic|municipio|cod_estado|dat_criaca|dat_atuali|
+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|       0|       0|         0|         0|       0|         0|       0|         0|        0|         0|         0|         8|
+--------+--------+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+

total de linhas: 1313653 | cod_imovel distintos: 1313620
+----------+-------+
|ind_status|  count|
+----------+-------+
|        AT|1305977|
|        PE|   4765|
|        CA|   2848|
|        SU|     63|
+----------+-------+

+-

In [6]:
# 5) investigar os cod_imovel duplicados
duplicados = df.groupBy("cod_imovel").count().filter("count > 1")
duplicados.orderBy(F.desc("count")).show(10, truncate=False)

duplicados_lista = [row["cod_imovel"] for row in duplicados.collect()]
df.filter(df.cod_imovel.isin(duplicados_lista)) \
  .select("cod_imovel", "ind_status", "des_condic", "dat_atuali") \
  .orderBy("cod_imovel") \
  .show(len(duplicados_lista) * 2, truncate=False)

# 6) checar outliers óbvios de área / módulo fiscal (negativos ou zerados)
df.select("mod_fiscal", "num_area").describe().show()
df.filter((F.col("num_area") <= 0) | (F.col("mod_fiscal") <= 0)).count()


+-------------------------------------------+-----+
|cod_imovel                                 |count|
+-------------------------------------------+-----+
|BA-2922854-20D021A9A1784E9B9F027190535F96DF|5    |
|BA-2924405-1B6E74379E8747D5BE460584C69500F1|3    |
|BA-2905107-A64B201026E54952ABB4DA3D2B28ABBE|3    |
|BA-2921500-EF449CC5A96A4D0FAE083721D39BF8F9|2    |
|BA-2903276-E8E63852360C4F9DB84A916E146AC3B4|2    |
|BA-2911105-2CB89F0B47474D26BD694DF314C8525F|2    |
|BA-2917359-94937D4ADA66494B97733A18E7D3C8B3|2    |
|BA-2928901-70550D3607B543E599923355CE566A7D|2    |
|BA-2917359-A898756B58294C72B9B72498F0D9A12F|2    |
|BA-2929107-EC17B963897C4B53B24338A83392FC53|2    |
+-------------------------------------------+-----+
only showing top 10 rows

+-------------------------------------------+----------+-------------------------+----------+
|cod_imovel                                 |ind_status|des_condic               |dat_atuali|
+-------------------------------------------+----------+--

173333

In [8]:
from pyspark.sql import Window

# cada grupo duplicado tem uma linha "AT" (ativa) + uma ou mais canceladas por
# duplicidade: ficamos com a linha AT quando existe, senão a mais recente por dat_atuali
janela = Window.partitionBy("cod_imovel").orderBy(
    F.when(F.col("ind_status") == "AT", 0).otherwise(1),
    F.desc("dat_atuali")
)

df_dedup = (
    df.withColumn("rn", F.row_number().over(janela))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

print(f"antes: {df.count()} | depois: {df_dedup.count()}")
print(f"cod_imovel distintos depois: {df_dedup.select('cod_imovel').distinct().count()}")


antes: 1313653 | depois: 1313620
cod_imovel distintos depois: 1313620


In [14]:
# descartar colunas constantes, preencher nulos e converter datas
# (o coalesce precisa acontecer ANTES da conversão pra date, enquanto as duas
# colunas ainda são texto)
df_limpo = (
    df_dedup
    .drop("cod_tema", "nom_tema")
    .withColumn("dat_atuali", F.coalesce(F.col("dat_atuali"), F.col("dat_criaca")))
    .withColumn("dat_criaca", F.to_date(F.col("dat_criaca"), "dd/MM/yyyy"))
    .withColumn("dat_atuali", F.to_date(F.col("dat_atuali"), "dd/MM/yyyy"))
)

df_limpo.printSchema()
df_limpo.select("dat_criaca", "dat_atuali").show(5)

# checagem final: não deve sobrar nenhum nulo
df_limpo.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_limpo.columns]).show()


root
 |-- cod_imovel: string (nullable = true)
 |-- mod_fiscal: double (nullable = true)
 |-- num_area: double (nullable = true)
 |-- ind_status: string (nullable = true)
 |-- ind_tipo: string (nullable = true)
 |-- des_condic: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- cod_estado: string (nullable = true)
 |-- dat_criaca: date (nullable = true)
 |-- dat_atuali: date (nullable = true)

+----------+----------+
|dat_criaca|dat_atuali|
+----------+----------+
|2026-06-09|2026-06-09|
|2017-11-02|2017-11-02|
|2017-11-02|2017-11-02|
|2022-11-23|2022-11-23|
|2022-07-16|2022-07-16|
+----------+----------+
only showing top 5 rows

+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|cod_imovel|mod_fiscal|num_area|ind_status|ind_tipo|des_condic|municipio|cod_estado|dat_criaca|dat_atuali|
+----------+----------+--------+----------+--------+----------+---------+----------+----------+----------+
|         0|         

## Etapa 3 — Análise Exploratória com Spark SQL

Registrar o DataFrame como view e responder pelo menos 5 perguntas de negócio.

In [24]:
df.createOrReplaceTempView("car")

### Pergunta 1


In [17]:
# Como é a distribuição de propriedades por município?
spark.sql("""
SELECT municipio, COUNT(*) AS qtd_imoveis, SUM(num_area) AS area_total, ROUND(AVG(num_area), 2) AS area_media
FROM car
GROUP BY municipio
ORDER BY area_total DESC
LIMIT 10
""").show()

+--------------------+-----------+------------------+----------+
|           municipio|qtd_imoveis|        area_total|area_media|
+--------------------+-----------+------------------+----------+
|Formosa do Rio Preto|       5509|1910207.3220999925|    346.74|
|       Sao Desiderio|       7716| 1511448.203100001|    195.88|
|          Correntina|       8182|1186648.1343000033|    145.03|
|        Pilao Arcado|      16224|1057719.5164000005|     65.19|
|           Jaborandi|       4273| 975111.2001999909|     228.2|
|            Sento Se|       6599| 968937.0880999957|    146.83|
|               Cocos|       3625| 907398.1850000018|    250.32|
|           Barreiras|       6282| 756953.9417999998|     120.5|
|               Barra|       8887| 756451.8793000021|     85.12|
|   Riachao das Neves|       3177| 648120.3966000018|     204.0|
+--------------------+-----------+------------------+----------+



### Pergunta 2


In [20]:
# Qual a distribuição das propriedades por tamanho?
spark.sql("""
SELECT
  CASE WHEN mod_fiscal <= 4 THEN 'Pequena'
       WHEN mod_fiscal <= 15 THEN 'Média'
       ELSE 'Grande' END AS faixa,
  COUNT(*) AS qtd_imoveis,
  SUM(num_area) AS area_total,
  ROUND(AVG(num_area), 2) AS area_media
FROM car
GROUP BY 1
ORDER BY area_total DESC
""").show()

+-------+-----------+--------------------+----------+
|  faixa|qtd_imoveis|          area_total|area_media|
+-------+-----------+--------------------+----------+
|Pequena|    1283109| 1.807189701499996E7|     14.08|
| Grande|       7250|1.5596481837599996E7|   2151.24|
|  Média|      23294|        8520160.4014|    365.77|
+-------+-----------+--------------------+----------+



### Pergunta 3


In [30]:
# Qual a distribuição por situação cadastral?
spark.sql("""
SELECT 
    CASE WHEN ind_status = 'AT' THEN 'Ativo'
         WHEN ind_status = 'PE' THEN 'Pendente'
         WHEN ind_status = 'CA' THEN 'Cancelado'
         WHEN ind_status = 'SU' THEN 'Suspenso'
        END AS status, COUNT(*) AS qtd, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
FROM car
GROUP BY ind_status
ORDER BY qtd DESC
""").show()

+---------+-------+----------+
|   status|    qtd|percentual|
+---------+-------+----------+
|    Ativo|1305977|     99.42|
| Pendente|   4765|      0.36|
|Cancelado|   2848|      0.22|
| Suspenso|     63|      0.00|
+---------+-------+----------+



### Pergunta 4


In [37]:
# Como foi a adesão ao CAR ao longo dos anos?
spark.sql("""
SELECT 
    YEAR(to_date(dat_criaca, 'dd/MM/yyyy')) AS ano, 
    COUNT(*) AS qtd_cadastros, 
    ROUND(SUM(num_area), 2) AS area_cadastrada
FROM car
WHERE dat_criaca IS NOT NULL
GROUP BY 1
ORDER BY 1
""").show()

+----+-------------+---------------+
| ano|qtd_cadastros|area_cadastrada|
+----+-------------+---------------+
|2014|         3239|     1229837.52|
|2015|        22604|     4738503.71|
|2016|        70576|     4785811.84|
|2017|       301554|     5207682.76|
|2018|       211564|     4388495.03|
|2019|       161670|     4073911.56|
|2020|        93801|     3683553.64|
|2021|        89481|     3197289.59|
|2022|        68293|     2493330.28|
|2023|        76555|     2925777.94|
|2024|        72696|     2282582.33|
|2025|        87120|     1882520.08|
|2026|        54500|     1299242.98|
+----+-------------+---------------+



### Pergunta 5


In [41]:
#Qual o tamanho médio das propriedades por tipo de imóvel
spark.sql("""
SELECT 
    CASE WHEN ind_tipo = 'IRU' THEN 'Propiedade Rural'
         WHEN ind_tipo = 'AST' THEN 'Assentamento'
         WHEN ind_tipo = 'PCT' THEN 'Povos/Comunidades Tradicionais'
    END AS tipo, COUNT(*) AS qtd, ROUND(AVG(num_area), 2) AS area_media, ROUND(AVG(mod_fiscal), 2) AS mod_fiscal_medio
FROM car
GROUP BY ind_tipo
ORDER BY qtd DESC
""").show()

+--------------------+-------+----------+----------------+
|                tipo|    qtd|area_media|mod_fiscal_medio|
+--------------------+-------+----------+----------------+
|    Propiedade Rural|1312087|     29.22|            0.57|
|        Assentamento|    905|   2834.72|           34.08|
|Povos/Comunidades...|    661|    1936.6|           17.65|
+--------------------+-------+----------+----------------+



## Etapa 4 — Modelagem Preditiva

Problema (classificação/regressão), 2+ modelos de famílias diferentes (1 ensemble), métricas e validação.

## Etapa 5 — Modelagem Descritiva

Clusterização (ex. K-Means), escolha do k (cotovelo/silhueta) e interpretação dos perfis.

## Etapa 6 — Interpretação e Conclusões (KDD)

Conhecimento descoberto, decisões possíveis, limitações e próximos passos.